[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rl-textbook/notebooks/blob/main/ch15_reasoning.ipynb)

<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #e94560; font-size: 2.2em; margin: 0;">Chapter 15: Reasoning, Tool Use & Reinforcement Finetuning</h1>
  <p style="color: #a8b2d8; font-size: 1.1em; margin-top: 10px;">Chain-of-Thought Prompting · Majority Voting · Inference-Time Scaling · Tool-Calling Loops</p>
</div>

This notebook demonstrates the key ideas from Chapter 15:
- **Chain-of-thought prompting** with a small/mock language model
- **Majority voting** (self-consistency): generate K solutions, take the most common answer
- **Accuracy vs. number of samples** — inference-time compute scaling
- **Tool-calling loops**: model emits JSON tool calls, code executes them


## Environment Setup

> **Colab/Kaggle**: This notebook runs on a free CPU runtime. No GPU is required.
> All LLM calls are **mocked** with a stochastic simulator so the notebook is fully self-contained.
> Wall-time estimate: ~2 minutes end-to-end.

In [ ]:
# !pip install numpy matplotlib --quiet  # already available on Colab
import numpy as np
import matplotlib.pyplot as plt
import json
import re
import random
from collections import Counter
from typing import List, Dict, Tuple, Optional

np.random.seed(42)
random.seed(42)
print("Imports OK")

<div style="background: #1e3a5f; padding: 16px; border-radius: 8px; border-left: 4px solid #4fc3f7;">
  <h2 style="color: #4fc3f7; margin: 0;">Part 1 — Chain-of-Thought Prompting</h2>
</div>

In **chain-of-thought (CoT) prompting** the model is asked to reason step-by-step before giving the final answer.  
We mock an LLM that solves arithmetic questions and either reasons correctly (high prob) or makes an error.

In [ ]:
# ----- Mock LLM -----
class MockLLM:
    """Simulates a small language model with controllable accuracy."""

    def __init__(self, correct_prob: float = 0.6, cot_correct_prob: float = 0.8):
        self.correct_prob = correct_prob
        self.cot_correct_prob = cot_correct_prob

    def _noisy_answer(self, correct: int, prob: float) -> int:
        if random.random() < prob:
            return correct
        # Return a plausible wrong answer
        return correct + random.choice([-2, -1, 1, 2, 3, 4, 5])

    def direct_answer(self, question: str, correct: int) -> Tuple[str, int]:
        """No CoT — just emit an answer."""
        ans = self._noisy_answer(correct, self.correct_prob)
        return f"Answer: {ans}", ans

    def cot_answer(self, question: str, correct: int) -> Tuple[str, int]:
        """CoT — emit reasoning then answer."""
        ans = self._noisy_answer(correct, self.cot_correct_prob)
        reasoning = (
            f"Let me think step by step.\n"
            f"  Step 1: Parse the problem.\n"
            f"  Step 2: Compute intermediate values.\n"
            f"  Step 3: Combine results → {ans}\n"
            f"Answer: {ans}"
        )
        return reasoning, ans


llm = MockLLM(correct_prob=0.55, cot_correct_prob=0.80)

questions = [
    ("What is 17 × 8?",           136),
    ("If x² = 49, what is x?",    7),
    ("43 + 58 = ?",               101),
    ("200 / 8 = ?",               25),
    ("3³ + 4² = ?",               43),
]

# Run one sample per mode
print("=== Direct (no CoT) ===")
for q, ans in questions[:3]:
    resp, pred = llm.direct_answer(q, ans)
    print(f"Q: {q}")
    print(f"  → {resp}  (correct={ans}, {'✓' if pred==ans else '✗'})")

print("\n=== Chain-of-Thought ===")
for q, ans in questions[:3]:
    resp, pred = llm.cot_answer(q, ans)
    print(f"Q: {q}")
    for line in resp.split('\n'):
        print(f"  {line}")
    print(f"  Correct={ans}, {'✓' if pred==ans else '✗'}")

In [ ]:
# Accuracy comparison: Direct vs CoT across many trials
N_TRIALS = 200
direct_correct = 0
cot_correct = 0

for _ in range(N_TRIALS):
    q, ans = random.choice(questions)
    _, p1 = llm.direct_answer(q, ans)
    _, p2 = llm.cot_answer(q, ans)
    direct_correct += int(p1 == ans)
    cot_correct   += int(p2 == ans)

print(f"Direct accuracy : {direct_correct/N_TRIALS:.1%}")
print(f"CoT accuracy    : {cot_correct/N_TRIALS:.1%}")

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Direct', 'Chain-of-Thought'], 
              [direct_correct/N_TRIALS, cot_correct/N_TRIALS],
              color=['#e94560', '#4fc3f7'], edgecolor='white', linewidth=1.5)
for bar, v in zip(bars, [direct_correct/N_TRIALS, cot_correct/N_TRIALS]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{v:.1%}', ha='center', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.0)
ax.set_ylabel('Accuracy')
ax.set_title('Direct vs Chain-of-Thought Accuracy', fontsize=13)
ax.set_facecolor('#0f1117')
fig.patch.set_facecolor('#0f1117')
ax.tick_params(colors='white'); ax.yaxis.label.set_color('white')
ax.title.set_color('white')
plt.tight_layout()
plt.show()

<div style="background: #1e3a5f; padding: 16px; border-radius: 8px; border-left: 4px solid #81c784;">
  <h2 style="color: #81c784; margin: 0;">Part 2 — Majority Voting (Self-Consistency)</h2>
</div>

**Self-consistency** (Wang et al., 2022): sample K independent chain-of-thought solutions and take the **majority answer**.  
This is a form of inference-time compute scaling — accuracy improves as K grows.

In [ ]:
def majority_vote(llm: MockLLM, question: str, correct: int, k: int) -> Tuple[int, bool]:
    """Generate k CoT samples and return the majority-vote answer."""
    answers = []
    for _ in range(k):
        _, ans = llm.cot_answer(question, correct)
        answers.append(ans)
    # Majority vote
    voted = Counter(answers).most_common(1)[0][0]
    return voted, (voted == correct)


# Demonstrate on a single question
q, ans = "What is 17 × 8?", 136
print(f"Question: {q}  (correct={ans})\n")
for k in [1, 3, 7, 15, 31]:
    # Average over 100 independent runs
    acc = np.mean([majority_vote(llm, q, ans, k)[1] for _ in range(100)])
    print(f"  k={k:2d} → majority-vote accuracy = {acc:.1%}")

In [ ]:
# Inference-time scaling: accuracy vs k (number of samples)
# ~30 s on CPU
K_VALUES = [1, 2, 3, 5, 7, 10, 15, 20, 31, 51]
N_REPEATS = 150  # independent accuracy estimates

mean_acc = []
std_acc  = []

for k in K_VALUES:
    trials = []
    for _ in range(N_REPEATS):
        q, ans = random.choice(questions)
        _, correct = majority_vote(llm, q, ans, k)
        trials.append(float(correct))
    mean_acc.append(np.mean(trials))
    std_acc.append(np.std(trials) / np.sqrt(N_REPEATS))

# Theoretical single-sample accuracy
p = llm.cot_correct_prob  # 0.80

# Theoretical majority-vote curve (for odd k)
from scipy.stats import binom
import warnings; warnings.filterwarnings('ignore')

def mv_accuracy(k, p):
    """Prob(majority of k Bernoulli(p) > k/2)."""
    return 1 - binom.cdf(k // 2, k, p)

k_fine = np.arange(1, 55)
theory = [mv_accuracy(k, p) for k in k_fine]

fig, ax = plt.subplots(figsize=(8, 5))
ax.fill_between(K_VALUES,
                np.array(mean_acc) - 2*np.array(std_acc),
                np.array(mean_acc) + 2*np.array(std_acc),
                alpha=0.2, color='#4fc3f7', label='±2 SE (empirical)')
ax.plot(K_VALUES, mean_acc, 'o-', color='#4fc3f7', lw=2, label='Empirical majority vote')
ax.plot(k_fine, theory, '--', color='#81c784', lw=2, label=f'Theory (p={p})')
ax.axhline(p, color='#e94560', linestyle=':', lw=1.5, label=f'Single-sample acc = {p}')
ax.set_xlabel('Number of samples K', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Inference-Time Scaling via Majority Voting', fontsize=13)
ax.legend(fontsize=10)
ax.set_ylim(0.5, 1.01)
ax.set_facecolor('#0f1117'); fig.patch.set_facecolor('#0f1117')
for item in [ax.xaxis.label, ax.yaxis.label, ax.title] + ax.get_xticklabels() + ax.get_yticklabels():
    item.set_color('white')
ax.legend(facecolor='#1a1a2e', labelcolor='white')
plt.tight_layout()
plt.show()
print("Key insight: accuracy monotonically increases with K when p > 0.5 (majority-vote theorem).")

<div style="background: #1e3a5f; padding: 16px; border-radius: 8px; border-left: 4px solid #ffb74d;">
  <h2 style="color: #ffb74d; margin: 0;">Part 3 — Best-of-N vs Majority Vote</h2>
</div>

Compare two inference-time strategies:
- **Best-of-N (oracle)**: pick the correct answer if any sample is correct (upper bound, requires a verifier)
- **Majority vote**: pick the most frequent answer (no verifier needed)

In [ ]:
def best_of_n(llm, question, correct, k):
    """Oracle: correct if any of k samples is correct."""
    for _ in range(k):
        _, ans = llm.cot_answer(question, correct)
        if ans == correct:
            return True
    return False


K_VALUES2 = [1, 3, 5, 10, 20, 40]
N_REP2 = 200

bon_acc, mv_acc2 = [], []
for k in K_VALUES2:
    bon_trials, mv_trials = [], []
    for _ in range(N_REP2):
        q, ans = random.choice(questions)
        bon_trials.append(float(best_of_n(llm, q, ans, k)))
        mv_trials.append(float(majority_vote(llm, q, ans, k)[1]))
    bon_acc.append(np.mean(bon_trials))
    mv_acc2.append(np.mean(mv_trials))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(K_VALUES2, bon_acc, 's--', color='#ffb74d', lw=2, label='Best-of-N (oracle)')
ax.plot(K_VALUES2, mv_acc2, 'o-',  color='#4fc3f7', lw=2, label='Majority Vote')
ax.axhline(llm.cot_correct_prob, color='#e94560', linestyle=':', lw=1.5, label='Single-sample')
ax.set_xlabel('Number of samples K')
ax.set_ylabel('Accuracy')
ax.set_title('Best-of-N (Oracle) vs Majority Vote', fontsize=13)
ax.legend(); ax.set_ylim(0.5, 1.01)
ax.set_facecolor('#0f1117'); fig.patch.set_facecolor('#0f1117')
for item in [ax.xaxis.label, ax.yaxis.label, ax.title] + ax.get_xticklabels() + ax.get_yticklabels():
    item.set_color('white')
ax.legend(facecolor='#1a1a2e', labelcolor='white')
plt.tight_layout()
plt.show()

<div style="background: #1e3a5f; padding: 16px; border-radius: 8px; border-left: 4px solid #ce93d8;">
  <h2 style="color: #ce93d8; margin: 0;">Part 4 — Tool-Calling Loop</h2>
</div>

Modern LLMs can call external tools by emitting structured JSON.  
We implement a minimal **ReAct-style** loop:
1. Model sees a question and available tool specs.
2. Model emits a JSON tool call.
3. The environment executes the tool and returns a result.
4. Model uses the result to produce the final answer.


In [ ]:
# ---- Tool definitions ----
TOOLS = {
    "calculator": {
        "description": "Evaluate a mathematical expression.",
        "params": {"expression": "string — a Python-evaluable math expression"},
        "fn": lambda expr: eval(expr, {"__builtins__": {}}, {})
    },
    "lookup": {
        "description": "Look up a fact from a knowledge base.",
        "params": {"query": "string — the fact to look up"},
        "fn": lambda q: {
            "speed of light": "299,792,458 m/s",
            "pi": "3.14159265",
            "gravitational constant": "9.81 m/s²",
        }.get(q.lower(), "Not found")
    }
}

def execute_tool(tool_call: Dict) -> str:
    """Execute a tool call dict {name, params} and return result as string."""
    name = tool_call.get("tool")
    if name not in TOOLS:
        return f"Error: unknown tool '{name}'"
    try:
        result = TOOLS[name]["fn"](**tool_call.get("params", {}))
        return str(result)
    except Exception as e:
        return f"Error: {e}"


# ---- Mock LLM with tool-calling ----
class ToolCallingLLM:
    """Simulates a model that emits tool calls in a ReAct loop."""

    def __init__(self):
        # Pre-scripted 'responses' keyed by question snippet
        self.plans = {
            "17 * 8": [
                {"thought": "I need to calculate 17 × 8.",
                 "tool": "calculator", "params": {"expression": "17 * 8"}},
            ],
            "sqrt": [
                {"thought": "I need sqrt(144).",
                 "tool": "calculator", "params": {"expression": "144 ** 0.5"}},
            ],
            "speed of light": [
                {"thought": "Look up the speed of light.",
                 "tool": "lookup", "params": {"query": "speed of light"}},
            ],
            "pi * 5**2": [
                {"thought": "Look up pi first.",
                 "tool": "lookup", "params": {"query": "pi"}},
                {"thought": "Now compute area = pi * r².",
                 "tool": "calculator", "params": {"expression": "3.14159265 * 5**2"}},
            ],
        }

    def plan(self, question: str) -> List[Dict]:
        for key, steps in self.plans.items():
            if key in question:
                return steps
        return [{"thought": "I'll try to compute directly.",
                 "tool": "calculator", "params": {"expression": "42"}}]


def react_loop(llm_agent: ToolCallingLLM, question: str) -> str:
    """Run the full ReAct loop and return the final answer."""
    print(f"\nQuestion: {question}")
    print("-" * 50)
    steps = llm_agent.plan(question)
    last_result = None
    for i, step in enumerate(steps):
        print(f"[Thought {i+1}] {step['thought']}")
        tool_call = {"tool": step["tool"], "params": step["params"]}
        print(f"[Action  {i+1}] {json.dumps(tool_call)}")
        result = execute_tool(tool_call)
        last_result = result
        print(f"[Observe {i+1}] {result}")
    print(f"[Answer] {last_result}")
    return last_result


agent = ToolCallingLLM()
react_loop(agent, "What is 17 * 8?")
react_loop(agent, "What is sqrt(144)?")
react_loop(agent, "What is the speed of light?")
react_loop(agent, "What is the area of a circle with radius 5? Use pi * 5**2")

In [ ]:
# Visualise: how tool use helps vs no-tool baseline
# Simulate: without tool, model has 40% chance on numeric questions; with tool, near-perfect

categories = ['Arithmetic\n(no tool)', 'Arithmetic\n(calculator)', 
               'Fact lookup\n(no tool)', 'Fact lookup\n(lookup tool)']
accuracies  = [0.40, 0.99, 0.30, 0.95]
colors      = ['#e94560', '#4fc3f7', '#e94560', '#4fc3f7']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(categories, accuracies, color=colors, edgecolor='white', linewidth=1.2)
for bar, v in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{v:.0%}', ha='center', fontsize=11, color='white', fontweight='bold')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Accuracy')
ax.set_title('Tool Use Dramatically Improves LLM Performance on Specific Tasks', fontsize=12)
ax.set_facecolor('#0f1117'); fig.patch.set_facecolor('#0f1117')
for item in [ax.xaxis.label, ax.yaxis.label, ax.title] + ax.get_xticklabels() + ax.get_yticklabels():
    item.set_color('white')
plt.tight_layout()
plt.show()

<div style="background: #1e3a5f; padding: 16px; border-radius: 8px; border-left: 4px solid #ef9a9a;">
  <h2 style="color: #ef9a9a; margin: 0;">Part 5 — RLVR Reward Signal (Simulated)</h2>
</div>

**Reinforcement Learning with Verifiable Rewards (RLVR)** trains a model using exact verifiers instead of reward models.  
We simulate how reward signal changes during training as the model improves.

In [ ]:
def simulate_rlvr_training(steps: int = 200, initial_acc: float = 0.3,
                            final_acc: float = 0.85, noise: float = 0.08) -> np.ndarray:
    """Simulate accuracy curve during RLVR training (sigmoid growth + noise)."""
    t = np.linspace(-3, 3, steps)
    acc = initial_acc + (final_acc - initial_acc) / (1 + np.exp(-t))
    acc += np.random.normal(0, noise, steps)
    return np.clip(acc, 0, 1)


np.random.seed(0)
steps = 300
acc_no_cot   = simulate_rlvr_training(steps, 0.25, 0.60, 0.06)
acc_cot_rlvr = simulate_rlvr_training(steps, 0.30, 0.85, 0.05)

# Smooth for plotting
def smooth(x, w=15): return np.convolve(x, np.ones(w)/w, mode='valid')

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(smooth(acc_cot_rlvr)))
ax.plot(x, smooth(acc_no_cot),   color='#e94560', lw=2, label='Baseline (no CoT)')
ax.plot(x, smooth(acc_cot_rlvr), color='#4fc3f7', lw=2, label='CoT + RLVR')
ax.fill_between(x, smooth(acc_cot_rlvr) - 0.04, smooth(acc_cot_rlvr) + 0.04,
                alpha=0.15, color='#4fc3f7')
ax.set_xlabel('Training Step')
ax.set_ylabel('Pass@1 Accuracy')
ax.set_title('Simulated RLVR Training Curve', fontsize=13)
ax.legend()
ax.set_facecolor('#0f1117'); fig.patch.set_facecolor('#0f1117')
for item in [ax.xaxis.label, ax.yaxis.label, ax.title] + ax.get_xticklabels() + ax.get_yticklabels():
    item.set_color('white')
ax.legend(facecolor='#1a1a2e', labelcolor='white')
plt.tight_layout()
plt.show()

## Summary & Key Takeaways

| Concept | Key Result |
|---------|------------|
| Chain-of-Thought | Step-by-step reasoning increases accuracy vs. direct answers |
| Majority Voting | Accuracy → 1 as K → ∞ when per-sample p > 0.5 |
| Best-of-N | Upper bounds majority vote; requires a verifier |
| Tool-calling | Near-perfect accuracy on tasks outsourced to exact tools |
| RLVR | Verifiable rewards allow RL training without a reward model |

**Inference-time scaling** is a powerful paradigm: by spending more compute at test time (more samples, longer chains of thought, tool calls), models can dramatically improve their effective accuracy — without changing model weights.